<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Batch-process for TSM retrieval

In this notebook we use the remote sensing reflectance data produced by the SNAP C2XComplex atmospheric correction as input for the TSM retrieval. Further information about the TSM algorithm can be found at [Jiang et al. (2023)](https://www.sciencedirect.com/science/article/pii/S0924271623002605)

#### This notebook does the following:
* creates a list of C2XComplex nc output files to process
* Runs the TSM retrieval algorithm
* Apply cloud and land mask as identified from Idepix
* reproject the TSM image, i.e., convert curvilinear to grid data, to correct the geolocation
* Saves output to netcdf

#### Requirements: 
* Make sure that you have already created some "*_C2XC_*.nc" output files using the *TSM_Step2_C2XComplex.ipynb* notebook.
* Make sure that you have set up the correct path for the shapefile that will be used to mask out land pixels.

#### Settings to adjust manually:
* Manually define the lake/dam name in code cell 4 (options: ZV, RV, TW, MV, VV, CW), default is TW. 

### Version history:
* Version 1.0, 23 Feb 2026

##### Authors:
**Dalin Jiang**, University of Stirling, UK

In [ ]:
import os
import glob
import time
from datetime import datetime
import xarray as xr
import numpy as np
import numba
from tqdm.auto import tqdm
from dask.distributed import Client
from scipy.interpolate import griddata
import geopandas as gpd
import rioxarray as rxr

In [ ]:
client = Client()
client

In [ ]:
# functions
#
@numba.njit
def Estimate_TSS_Jiang(site_Rrs):
    aw = np.array([0.00515124,0.01919594,0.06299986,0.41395333,0.70385758,2.71167020,2.62000141,4.61714226])
    bbw = np.array([0.00215037,0.00138116,0.00078491,0.00037474,0.00029185,0.00023499,0.00018516,0.00012066])
    wave =  np.array([443.0,490.0,560.0,665.0,705.0,740.0,783.0,865.0])

    def QAA_560(site_Rrs):
        rrs = site_Rrs/(0.52+1.7*site_Rrs)
        u = (-0.089+np.sqrt((0.089**2)+4*0.125*rrs))/(2*0.125)
        x = np.log10((rrs[0]+rrs[1])/(rrs[2]+5*rrs[3]*rrs[3]/rrs[1]))
        a560 = aw[2]+10.0**(-1.146-1.366*x-0.469*(x**2))
        bbp560 = ((u[2]*a560)/(1-u[2]))-bbw[2]
        one_tss = 40.35141492*bbp560
        return(one_tss)

    def QAA_665(site_Rrs):
        rrs = site_Rrs/(0.52+1.7*site_Rrs)
        u = (-0.089+np.sqrt((0.089**2)+4*0.125*rrs))/(2*0.125)
        a665 = aw[3]+0.39*((site_Rrs[3]/(site_Rrs[0]+site_Rrs[1]))**1.14)
        bbp665 = ((u[3]*a665)/(1-u[3]))-bbw[3]
        one_tss = 43.9677864*bbp665
        return(one_tss)    

    def QAA_740(site_Rrs):
        rrs = site_Rrs/(0.52+1.7*site_Rrs)
        u = (-0.089+np.sqrt((0.089**2)+4*0.125*rrs))/(2*0.125)
        bbp740 = ((u[5]*aw[5])/(1-u[5]))-bbw[5]
        one_tss = 46.41238517*bbp740
        return(one_tss) 

    def QAA_865(site_Rrs):
        rrs = site_Rrs/(0.52+1.7*site_Rrs)
        u = (-0.089+np.sqrt((0.089**2)+4*0.125*rrs))/(2*0.125)
        bbp865 = ((u[7]*aw[7])/(1-u[7]))-bbw[7]
        one_tss = 50.15399306*bbp865
        return(one_tss) 

    def estimate_Rrs620(in665):
        a = 1.693846e+02                
        b = -1.557556e+01                 
        c = 1.316727e+00                  
        d = 1.484814e-04                
	
        est620 = a*in665**3 + b*in665**2 + c*in665 + d	
        return(est620)

    Rrs620 = estimate_Rrs620(site_Rrs[3])

    if (np.all((np.isnan(site_Rrs)) | (site_Rrs == 0))):
        tmp_tss = np.nan
    elif (site_Rrs[1] > site_Rrs[2]):
        tmp_tss = QAA_560(site_Rrs)
    elif (site_Rrs[1] > Rrs620):
        tmp_tss = QAA_665(site_Rrs)    
    elif ((site_Rrs[5] > site_Rrs[1]) & (site_Rrs[5] > 0.010)):
        tmp_tss = QAA_865(site_Rrs)      
    else:
        tmp_tss = QAA_740(site_Rrs)

    return(tmp_tss)


def convert_curvilinear_to_grid(ds, var_name):
    # Flatten curvilinear coordinates and data
    one_var = ds[var_name]
    geo_pts = np.column_stack([ds.lon.values.ravel(), ds.lat.values.ravel()])
    var_dt = one_var.values.ravel() 

    # define the size of output file
    xsize = one_var.shape[1]  # width, lon
    ysize = one_var.shape[0]  # height, lat

    # define pixel size of output
    #pixel_size = 0.00018  # apprximate 20 m
    
    # Define target grid
    lon_new, lat_new = np.meshgrid(
        np.linspace(ds.lon.min(), ds.lon.max(), xsize),
        np.linspace(ds.lat.min(), ds.lat.max(), ysize)
    )
    # Interpolate (linear/cubic/nearest)
    regridded_dt = griddata(
        geo_pts, var_dt,
        (lon_new, lat_new),
        method='linear'
    )
    
    # Create new xarray Dataset
    ds_new = xr.Dataset(
        {var_name: (('y', 'x'), regridded_dt)}, # y=lat, x=lon
        coords={
            'y': lat_new[:,0],
            'x': lon_new[0,:]
        }
    )
    
    return(ds_new)


def crop_lake_and_add_info(ds, lake_shape, lake_name):
    """
    ds: the raw image to be cropped
    lake_shape: the shape file containing all study lakes
    lake_name: short name of the lake, to get a single lake shape from the shape file, i.e., ZV, TW, RV
    """
    ds = ds.rio.write_crs("EPSG: 4326")
    lake_roi = lake_shape[lake_shape["Short_name"] == lake_name]
    mk_arr = ds["TSM"].rio.clip(lake_roi.geometry.values, drop=True)
    mk_dt = mk_arr.to_dataset(name="TSM")

    # convert data to float32 to save space
    for var in ['x','y','TSM']:
        mk_dt[var] = mk_dt[var].astype('float32')

    # change coordinates name, otherwise SNAP won't read correctly
    mk_dt = mk_dt.rename({'y':'lat', 'x':'lon'})
    
    # add attributes
    mk_dt["TSM"].attrs={'standard_name':'Total Suspended Matter',
                        'short_name':'TSM',
                       'units': 'g m^-3',
                       'grid_mapping':'spatial_ref'}

    # add global attributes
    mk_dt.attrs = {'Product': 'Total Suspended Matter',
                   'Units': 'g m^-3',
                   'Sensor': 'Sentinel2-MSI',
                   'AC Algorithm': 'C2XComplex',
                   'AC Algorithm DOI': 'Brockmann et al. (2016)',
                   'TSM algorithm': 'Jiang et al. (2023)',
                   'TSM algorithm DOI': 'https://doi.org/10.1016/j.isprsjprs.2023.09.020',
                   'Project': 'ESA EO Africa R&D SWAM',
                   'Project PI':'Marie Smith, CSIR; Dalin Jiang, University of Stirling',
                   'PI contact':'msmith2@csir.co.za; dalin.jiang@stir.ac.uk'
                   }

    return(mk_dt)


def mask_cloud_risk(ds, wq_dt):
    """
    To mask out pixels with cloud risk
    input: (1)ds: Rrs DataSet from C2XComplex; (2) wq_dt: water quality DataArray, i.e., Chla, TSM
    output: cloud_risk masked water quality data
    
    """
    # Access flags and chlorophyll as Dask arrays (do NOT call .values)
    flags = ds['c2rcc_flags']
    
    # Define which flag bits to mask out
    mask_flags = [8]
    
    # Create bitwise mask using Dask array operations (lazy)
    cloud_mask = np.zeros(flags.shape, dtype=bool)
    for f in mask_flags:
        cloud_mask = cloud_mask | (flags & f != 0)
    
    # Invert to get valid pixels mask
    valid_mask = ~cloud_mask

    # Create mask for chlorophyll > 0 (lazy operation)
    data_mask = wq_dt > 0

    # Combine masks
    combined_mask = data_mask & valid_mask

    # Use combined_mask to mask chlorophyll (this remains lazy)
    masked_wq = wq_dt.where(combined_mask)

    return (masked_wq)

## Step 1: search for the atmospheric corrected images
the following things need to be set up:
* input path, i.e., the path include all the atmospheric corrected images
* output path, i.e., where the TSM results will be saved
* roi file: the shapefile of the lake, will be used to mask out land pixels

In [ ]:
# input/output path
in_path = "/media/dj16/DATA/satellite_image/Sentinel_2/South_Africa/atm_test/"
out_path = '/media/dj16/DATA/satellite_image/Sentinel_2/South_Africa/TSM_test/'

# lake shape file to crop the image
roi_file = "/media/dj16/DATA/GIS_data/South_Africa/EO_Africa_lake_polygon_updated.geojson"
roi = gpd.read_file(roi_file)

all_img = glob.glob(in_path+"*.nc",recursive=True)
print(f' found {len(all_img)} images, with the first 5 showing below:')
all_img[0:5]

## Step 2: run TSM estimation in batch
this include the following processing:
* estiamte TSM
* convert data into regular grid
* mask cloud and land pixels
* export results to nc files

In [ ]:
# process all images
for one_img in tqdm(all_img):
    start_time = time.time()
    
    one_dt = xr.open_dataset(one_img)        #,chunks={"x": 512, "y": 512}
    one_rrs = one_dt[['rrs_B1','rrs_B2','rrs_B3','rrs_B4','rrs_B5','rrs_B6','rrs_B7','rrs_B8A']].chunk({"x": 1000, "y": 1000})

    # get image information: name of lake, date and time
    img_info = os.path.basename(one_img)
    lake_name = img_info.split('_')[4].replace('.nc','')
    time_stamp = datetime.strptime(img_info.split('_')[1], "%Y%m%dT%H%M%S") 
    
    # convert to 3D array: band, x, y
    grd_dt = one_rrs.to_array(dim='band')
    grd_dt = grd_dt.chunk(dict(x=1000, y=1000, band=-1)) #band: -1 means all bands in one chunk — important for apply_ufunc.

    # prepare dask for running the calculation
    result = xr.apply_ufunc(
        Estimate_TSS_Jiang,
        grd_dt,  # 3D DataArray with dims ('band', 'y', 'x')
        input_core_dims=[["band"]], # along with dim to apply the function
        #output_core_dims=[],
        vectorize=True,
        dask="parallelized",    # use only if `da` is dask-backed
        output_dtypes=[float]
    )
    # trigger the computing
    raw_result = result.compute()
    raw_dt = mask_cloud_risk(ds=one_dt, wq_dt=raw_result).to_dataset(name="TSM") # mask cloud risk and convert to dataset

    # convert the data to a regular grid 
    reg_dt = convert_curvilinear_to_grid(ds=raw_dt, var_name='TSM')
    water_dt = crop_lake_and_add_info(ds=reg_dt, lake_shape=roi, lake_name=lake_name)
    water_dt = water_dt.expand_dims(dim={'time':[time_stamp]})
    water_dt.attrs["start_date"] = one_dt.attrs["start_date"]  # copy information
    water_dt.attrs["stop_date"] = one_dt.attrs["stop_date"]
    
    comp = dict(zlib=True, complevel=6, dtype='float32')  # 1–9 (higher = more compression, slower)
    encoding = {var: comp for var in water_dt.data_vars}

    out_file = out_path + img_info.replace(".nc","_TSM_update.nc")
    water_dt.to_netcdf(out_file,format='NETCDF4', encoding=encoding)

    time_used = time.time() - start_time
    print(f"processing {img_info } used {time_used.3f} seconds" 